# Negative Binomial Model for Cyclist Volume in Adelaide

## Variables:
- POW_CTW: N people who work in this DZN and cycle to work
    - ```all-data/census-formatted/greater-adelaide/MTW-Bicycle-POW-DZN.csv``` [Bicycle]
- POW_FE: N people who work in this DZN and are fullly engaged in employment or study
    - ```all-data/census-formatted/greater-adelaide/EETP-POW-DZN.csv``` [Fully Engaged]
- INF_CW: 1 if the edge the count point lies on is a cycleway, else 0
- MB population density: PD_MB
- Distance to nearest large DZN: DNL_DZN
- Age demographics, try males 30-40 and females 15-25 (best from Dadshova's): M_30_40, F_15_25
- Households with a THI of > 3500, place of enumeration, MB: H_3500
- OSM highway=trunk: INF_LRG_RD
- OSM highway={primary, secondary}: INF_MED_RD
- OSM highway={tertiary, residential}: INF_SML_RD
- OSM cycleway:{left, right}=lane, not separate, just a normal bike lane: INF_CL
- Manual Overpass IS_CYCLE_ROUTE: INF_CR

## Correlation Risks:

Not sure yet we will see! Got lots of vars so removing some later wouldn't hurt

__*Generate Adelaide Graph Network*__

In [1]:
import osmnx as ox
import matplotlib.pyplot as plt
import geopandas as gpd

ox.settings.log_console = True
ox.settings.use_cache = True

ox.settings.useful_tags_way += [
    "cycleway",
    "cycleway:left",
    "cycleway:right",
    "cycleway:both",
    "bicycle",
]

gdf_boundary = ox.geocode_to_gdf("R11381689", by_osmid=True)
polygon = gdf_boundary.geometry.iloc[0]

G = ox.graph_from_polygon(polygon, network_type="bike")

__*Add in cycle route relations*__

In [6]:
import sys
sys.path.insert(0, '/Users/alexhuynh/Documents/AdelaideCyclistVolumesModel')
from infrastructure.cycle_routes import enrich_graph_with_cycle_routes
# Get the graph with IS_CYCLE_ROUTE edge attribute
G = enrich_graph_with_cycle_routes(G)

Found 2591 ways in bicycle routes


__*Import Street Network Data*__

In [7]:
import networkx as nx

infrastructure_tags = {
    "highway": [
        "motorway", "trunk",
        "primary", "secondary", "tertiary",
        "unclassified", "residential", "living_street",
        "service", "cycleway",
    ],
    "cycleway": True,
    "cycleway:left": True,
    "cycleway:right": True,
    "cycleway:both": True,
    "bicycle": True,
}

infrastructure_data = ox.features_from_polygon(
    polygon,
    tags=infrastructure_tags,
)

nodes_gdf, edges_gdf = ox.graph_to_gdfs(
    G,
    nodes=True,
    edges=True,
)

# Match infrastructure geometries to graph edges.
infrastructure_data = infrastructure_data[
    infrastructure_data.geometry.notna()
].to_crs(edges_gdf.crs).copy()
infrastructure_data["_has_infrastructure"] = True

edge_infrastructure_matches = gpd.sjoin(
    edges_gdf[["geometry"]],
    infrastructure_data[["_has_infrastructure", "geometry"]],
    how="left",
    predicate="intersects",
)

matched_edges = edge_infrastructure_matches.loc[
    edge_infrastructure_matches["_has_infrastructure"].eq(True)
].index.unique()

nx.set_edge_attributes(G, False, "HAS_BICYCLE_INFRASTRUCTURE")

for source, target, key in matched_edges:
    G.edges[source, target, key]["HAS_BICYCLE_INFRASTRUCTURE"] = True

print(f"Edges with infrastructure: {len(matched_edges)} / {len(edges_gdf)}")

ResponseStatusCodeError: 'overpass-api.de' responded: 413 Request Entity Too Large <html>
<head><title>413 Request Entity Too Large</title></head>
<body>
<center><h1>413 Request Entity Too Large</h1></center>
<hr><center>nginx</center>
</body>
</html>


__*Load MBs and DZNs into graph edges*__

In [ ]:
mbs_aus = gpd.read_file("../all-data/australia-mesh-blocks-raw/MB_2021_AUST_GDA2020.shp")
dzns_aus = gpd.read_file("../all-data/australia-destination-zones-raw/DZN_2021_AUST_GDA2020.shp")

mb_gdf_boundary_crs = gdf_boundary.to_crs(mbs_aus.crs)
dzn_gdf_boundary_crs = gdf_boundary.to_crs(dzns_aus.crs)
mbs = gpd.clip(mbs_aus, mb_gdf_boundary_crs)
dzns = gpd.clip(dzns_aus, dzn_gdf_boundary_crs)

print('MB all count: ', len(mbs_aus))
print('MB Adelaide count: ', len(mbs))
print('DZN all count: ', len(dzns_aus))
print('DZN Adelaide count: ', len(dzns))

In [ ]:
nodes_gdf = ox.graph_to_gdfs(G, nodes=True, edges=False)[["geometry"]]
 
nodes_mbs = gpd.sjoin(
    nodes_gdf.to_crs(mbs.crs),
    mbs[["MB_CODE21", "geometry"]],
    how="left",
    predicate="within",
)
node_mb_codes = nodes_mbs["MB_CODE21"].dropna().to_dict()
nx.set_node_attributes(G, node_mb_codes, "MB_CODE21")

nodes_dzns = gpd.sjoin(
    nodes_gdf.to_crs(dzns.crs),
    dzns[["DZN_CODE21", "geometry"]],
    how="left",
    predicate="within",
)
node_dzn_codes = nodes_dzns["DZN_CODE21"].dropna().to_dict()
nx.set_node_attributes(G, node_dzn_codes, "DZN_CODE21")

print(f"Graph nodes: {len(G.nodes)}")
print(f"Nodes with MB_CODE21: {len(node_mb_codes)}")
print(f"Nodes with DZN_CODE21: {len(node_dzn_codes)}")

In [ ]:
import pandas as pd
nodes_to_remove = [
    node_id
    for node_id, node_data in G.nodes(data=True)
    if not (
        pd.notna(node_data.get("MB_CODE21"))
        or pd.notna(node_data.get("DZN_CODE21"))
    )
]

G.remove_nodes_from(nodes_to_remove)
print(f"Removed {len(nodes_to_remove)} nodes; remaining nodes: {len(G.nodes)}")
print(f"Remaining edges: {len(G.edges)}")

## Retrieve following vars for graph and regression table

__Generic Census Data:__
- DNL_DZN
- PD_MB

__Census Table Data:__
- POW_CTW
- POW_FE
- M_30_40
- F_15_25
- H_3500

__Infrastructure Data:__
- INF_LRG_RD
- INF_MED_RD
- INF_SML_RD
- INF_CL
- INF_CR
- INF_CW

__*DNL_DZN*__

Distance to nearest large destination zone (8000+ jobs). These zones include the areas near North Terrace / Rundle / Franklin st, the RAH, between Pirie and Wakefield St, East End. 8000 is an arbitrary number.